# 4η Εργασία - Βαθιά Μάθηση στο MNIST

**Μάθημα:** Τεχνητή Νοημοσύνη  
**Φοιτητής** Καρακέβας Νικόλαος  
**ΑΕΜ:** 5010 

Στόχος του notebook είναι η εκπαίδευση και αξιολόγηση Βαθιών Νευρωνικών Δικτύων για ταξινόμηση χειρόγραφων ψηφίων MNIST. Περιλαμβάνονται:

1. Προεπεξεργασία δεδομένων.
2. Πειράματα hyper-parameter tuning σε fully connected DNN.
3. Βελτίωση με διαφορετική αρχιτεκτονική CNN.
4. Πίνακες αποτελεσμάτων, διαγράμματα και confusion matrix.

In [ ]:
# Βασικές βιβλιοθήκες
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import confusion_matrix, classification_report

# Αναπαραγωγιμότητα πειραμάτων
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)

## 1. Φόρτωση και προεπεξεργασία δεδομένων

Το MNIST περιέχει ασπρόμαυρες εικόνες 28x28 pixel με ψηφία 0-9. Οι τιμές των pixel κανονικοποιούνται στο διάστημα [0, 1]. Για τα πλήρως συνδεδεμένα DNN οι εικόνες μετατρέπονται σε διανύσματα 784 τιμών, ενώ για το CNN διατηρείται η δισδιάστατη μορφή 28x28x1.

In [ ]:
# Φόρτωση MNIST από το Keras
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Κρατάμε validation set από το training set
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

x_val = x_train[-10000:]
y_val = y_train[-10000:]
x_train_small = x_train[:-10000]
y_train_small = y_train[:-10000]

# Μορφή για Dense DNN
x_train_dense = x_train_small.reshape((-1, 28 * 28))
x_val_dense = x_val.reshape((-1, 28 * 28))
x_test_dense = x_test.reshape((-1, 28 * 28))

# Μορφή για CNN
x_train_cnn = x_train_small[..., np.newaxis]
x_val_cnn = x_val[..., np.newaxis]
x_test_cnn = x_test[..., np.newaxis]

print('Training set:', x_train_small.shape, y_train_small.shape)
print('Validation set:', x_val.shape, y_val.shape)
print('Test set:', x_test.shape, y_test.shape)

In [ ]:
# Εμφάνιση ενδεικτικών εικόνων MNIST
plt.figure(figsize=(8, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train_small[i], cmap='gray')
    plt.title(f'Label: {y_train_small[i]}')
    plt.axis('off')
plt.tight_layout()
plt.show()

## 2. Βοηθητικές συναρτήσεις

Οι επόμενες συναρτήσεις δημιουργούν μοντέλα, τα εκπαιδεύουν και επιστρέφουν μετρήσεις απόδοσης. Χρησιμοποιείται `SparseCategoricalCrossentropy`, επειδή οι ετικέτες είναι ακέραιοι αριθμοί 0-9 και όχι one-hot vectors.

In [ ]:
def build_dense_model(hidden_layers=(128, 64), activation='relu', learning_rate=0.001, dropout_rate=0.0):
    """Δημιουργεί fully connected Deep Neural Network για MNIST."""
    model = keras.Sequential(name='Dense_DNN')
    model.add(layers.Input(shape=(784,)))

    for units in hidden_layers:
        model.add(layers.Dense(units, activation=activation))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(10, activation='softmax'))

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def train_and_evaluate_dense(config):
    """Εκπαιδεύει ένα Dense DNN με βάση ένα config και επιστρέφει αποτελέσματα."""
    model = build_dense_model(
        hidden_layers=config['hidden_layers'],
        activation=config['activation'],
        learning_rate=config['learning_rate'],
        dropout_rate=config.get('dropout_rate', 0.0)
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=2,
            restore_best_weights=True
        )
    ]

    history = model.fit(
        x_train_dense, y_train_small,
        validation_data=(x_val_dense, y_val),
        epochs=config['epochs'],
        batch_size=config['batch_size'],
        verbose=0,
        callbacks=callbacks
    )

    test_loss, test_acc = model.evaluate(x_test_dense, y_test, verbose=0)
    best_val_acc = max(history.history['val_accuracy'])

    return {
        'model': model,
        'history': history,
        'test_loss': test_loss,
        'test_accuracy': test_acc,
        'best_val_accuracy': best_val_acc,
        'epochs_ran': len(history.history['loss'])
    }

## 3. Hyper-parameter tuning

Δοκιμάζονται διαφορετικοί συνδυασμοί παραμέτρων: αριθμός επιπέδων, αριθμός νευρώνων, ρυθμός μάθησης, εποχές, συνάρτηση ενεργοποίησης, batch size και dropout. Το ζητούμενο δεν είναι απλώς να βρεθεί μία καλή ακρίβεια, αλλά να φανεί τεκμηριωμένα πώς επηρεάζεται η απόδοση από τις επιλογές υπερπαραμέτρων.

In [ ]:
experiments = [
    {
        'name': 'DNN_128_64_relu_lr1e-3',
        'hidden_layers': (128, 64),
        'activation': 'relu',
        'learning_rate': 0.001,
        'epochs': 10,
        'batch_size': 128,
        'dropout_rate': 0.0
    },
    {
        'name': 'DNN_256_128_relu_lr1e-3',
        'hidden_layers': (256, 128),
        'activation': 'relu',
        'learning_rate': 0.001,
        'epochs': 10,
        'batch_size': 128,
        'dropout_rate': 0.0
    },
    {
        'name': 'DNN_512_256_128_relu_lr1e-3',
        'hidden_layers': (512, 256, 128),
        'activation': 'relu',
        'learning_rate': 0.001,
        'epochs': 12,
        'batch_size': 128,
        'dropout_rate': 0.2
    },
    {
        'name': 'DNN_256_128_tanh_lr1e-3',
        'hidden_layers': (256, 128),
        'activation': 'tanh',
        'learning_rate': 0.001,
        'epochs': 10,
        'batch_size': 128,
        'dropout_rate': 0.0
    },
    {
        'name': 'DNN_256_128_relu_lr5e-4',
        'hidden_layers': (256, 128),
        'activation': 'relu',
        'learning_rate': 0.0005,
        'epochs': 12,
        'batch_size': 128,
        'dropout_rate': 0.0
    },
    {
        'name': 'DNN_256_128_relu_lr1e-3_bs64',
        'hidden_layers': (256, 128),
        'activation': 'relu',
        'learning_rate': 0.001,
        'epochs': 10,
        'batch_size': 64,
        'dropout_rate': 0.0
    }
]

results = []
trained_dense_models = {}

for config in experiments:
    print(f"Training: {config['name']}")
    output = train_and_evaluate_dense(config)
    trained_dense_models[config['name']] = output
    results.append({
        'Model': config['name'],
        'Hidden layers': str(config['hidden_layers']),
        'Activation': config['activation'],
        'Learning rate': config['learning_rate'],
        'Batch size': config['batch_size'],
        'Dropout': config['dropout_rate'],
        'Max epochs': config['epochs'],
        'Epochs ran': output['epochs_ran'],
        'Best val accuracy': output['best_val_accuracy'],
        'Test accuracy': output['test_accuracy'],
        'Test loss': output['test_loss']
    })

results_df = pd.DataFrame(results).sort_values('Test accuracy', ascending=False)
results_df.to_csv('hyperparameter_tuning_results.csv', index=False)
results_df

In [ ]:
# Διάγραμμα σύγκρισης test accuracy
plt.figure(figsize=(10, 5))
plt.bar(results_df['Model'], results_df['Test accuracy'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('Test accuracy')
plt.title('Σύγκριση ακρίβειας ανά πείραμα DNN')
plt.tight_layout()
plt.show()

best_dense_name = results_df.iloc[0]['Model']
best_dense_model = trained_dense_models[best_dense_name]['model']
print('Best Dense model:', best_dense_name)

## 4. Βελτίωση απόδοσης με CNN

Εκτός από απλό fully connected DNN, υλοποιείται Convolutional Neural Network. Η επιλογή CNN είναι κατάλληλη για εικόνες, επειδή οι συνελικτικές στρώσεις αξιοποιούν τη γειτονική σχέση των pixel και εντοπίζουν τοπικά μοτίβα, όπως γραμμές, καμπύλες και γωνίες. Αυτό είναι πλεονέκτημα σε σχέση με την απλή μετατροπή της εικόνας σε διάνυσμα 784 τιμών.

In [ ]:
def build_cnn_model(learning_rate=0.001, dropout_rate=0.25):
    model = keras.Sequential(name='Improved_CNN')
    model.add(layers.Input(shape=(28, 28, 1)))

    model.add(layers.Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D(pool_size=(2, 2)))
    model.add(layers.Dropout(dropout_rate))

    model.add(layers.Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D(pool_size=(2, 2)))
    model.add(layers.Dropout(dropout_rate))

    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(10, activation='softmax'))

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

cnn_model = build_cnn_model(learning_rate=0.001, dropout_rate=0.25)
cnn_model.summary()

In [ ]:
cnn_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        'best_cnn_model.keras',
        monitor='val_accuracy',
        save_best_only=True
    )
]

cnn_history = cnn_model.fit(
    x_train_cnn, y_train_small,
    validation_data=(x_val_cnn, y_val),
    epochs=15,
    batch_size=128,
    callbacks=cnn_callbacks,
    verbose=1
)

cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test_cnn, y_test, verbose=0)
print('CNN test loss:', cnn_test_loss)
print('CNN test accuracy:', cnn_test_acc)

In [ ]:
# Σύγκριση καλύτερου Dense DNN με το CNN
comparison_df = pd.DataFrame([
    {
        'Model': best_dense_name,
        'Type': 'Fully connected DNN',
        'Test accuracy': float(results_df.iloc[0]['Test accuracy'])
    },
    {
        'Model': 'Improved_CNN',
        'Type': 'Convolutional Neural Network',
        'Test accuracy': float(cnn_test_acc)
    }
])
comparison_df.to_csv('final_model_comparison.csv', index=False)
comparison_df

In [ ]:
# Καμπύλες εκπαίδευσης για το CNN
plt.figure(figsize=(8, 5))
plt.plot(cnn_history.history['accuracy'], label='Training accuracy')
plt.plot(cnn_history.history['val_accuracy'], label='Validation accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('CNN training / validation accuracy')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(cnn_history.history['loss'], label='Training loss')
plt.plot(cnn_history.history['val_loss'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('CNN training / validation loss')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Αξιολόγηση τελικού μοντέλου

Στο σημείο αυτό υπολογίζεται confusion matrix και classification report για το τελικό CNN. Αυτά βοηθούν να φανεί όχι μόνο η συνολική ακρίβεια, αλλά και σε ποιες κλάσεις κάνει τα περισσότερα λάθη το μοντέλο.

In [ ]:
y_pred_probs = cnn_model.predict(x_test_cnn, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_test, y_pred)
print(classification_report(y_test, y_pred, digits=4))

plt.figure(figsize=(7, 7))
plt.imshow(cm, interpolation='nearest')
plt.title('Confusion Matrix - CNN')
plt.colorbar()
plt.xticks(np.arange(10), np.arange(10))
plt.yticks(np.arange(10), np.arange(10))
plt.xlabel('Predicted label')
plt.ylabel('True label')

threshold = cm.max() / 2
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha='center', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# Εμφάνιση μερικών λανθασμένων προβλέψεων
wrong_idx = np.where(y_pred != y_test)[0]
print('Number of wrong predictions:', len(wrong_idx))

plt.figure(figsize=(10, 6))
for i, idx in enumerate(wrong_idx[:15]):
    plt.subplot(3, 5, i + 1)
    plt.imshow(x_test[idx], cmap='gray')
    plt.title(f'True: {y_test[idx]} | Pred: {y_pred[idx]}')
    plt.axis('off')
plt.tight_layout()
plt.show()

## 6. Ερωτήσεις κατανόησης

### α. Είναι τα δεδομένα MNIST καλά για εκπαίδευση μοντέλου;
Ναι, για εκπαιδευτικούς και πειραματικούς σκοπούς είναι πολύ καλό dataset, επειδή είναι καθαρό, ισορροπημένο, μικρό σε υπολογιστικό κόστος και περιέχει αρκετά δείγματα. Ωστόσο, δεν αντιπροσωπεύει πλήρως πιο δύσκολες πραγματικές συνθήκες, όπου οι εικόνες μπορεί να έχουν θόρυβο, διαφορετικό φωτισμό, φόντο ή παραμορφώσεις.

### β. Είναι όλα τα pixel σημαντικά για την πρόβλεψη;
Όχι το ίδιο. Τα κεντρικά pixel συνήθως μεταφέρουν περισσότερη πληροφορία, επειδή εκεί βρίσκεται το ψηφίο. Τα περιφερειακά pixel είναι συχνά μαύρα και έχουν μικρότερη συνεισφορά. Παρόλα αυτά, η σχετική θέση και οι γειτονικές σχέσεις των pixel έχουν σημασία, ειδικά σε CNN.

### γ. Πότε είναι καλή ιδέα να χρησιμοποιούνται Βαθιά Νευρωνικά Δίκτυα;
Είναι κατάλληλα όταν υπάρχουν πολλά δεδομένα, σύνθετα μοτίβα και ανάγκη αυτόματης εξαγωγής χαρακτηριστικών, όπως σε εικόνες, ήχο, φυσική γλώσσα, συστήματα πρόβλεψης και αναγνώριση προτύπων. Δεν είναι πάντα η καλύτερη επιλογή για μικρά datasets ή απλά προβλήματα, όπου απλούστερα μοντέλα είναι πιο ερμηνεύσιμα και οικονομικά.

### δ. Μπορεί η Βαθιά Μάθηση να χρησιμοποιηθεί σε supervised, unsupervised και reinforcement learning;
Ναι. Στο supervised learning χρησιμοποιείται για ταξινόμηση και παλινδρόμηση. Στο unsupervised learning χρησιμοποιείται σε autoencoders, clustering representations και generative models. Στο reinforcement learning χρησιμοποιείται σε Deep Reinforcement Learning, όπου νευρωνικά δίκτυα προσεγγίζουν πολιτικές ή συναρτήσεις αξίας.

## 7. Συμπέρασμα

Στην εργασία εκπαιδεύτηκαν αρχικά πλήρως συνδεδεμένα νευρωνικά δίκτυα με διαφορετικές υπερπαραμέτρους. Έπειτα υλοποιήθηκε CNN ως βελτίωση της αρχιτεκτονικής. Το CNN αναμένεται να αποδώσει καλύτερα στο MNIST, επειδή διατηρεί τη χωρική πληροφορία των εικόνων και μαθαίνει τοπικά χαρακτηριστικά. Μετά την εκτέλεση, χρησιμοποιήστε τους πίνακες `hyperparameter_tuning_results.csv` και `final_model_comparison.csv` για να συμπληρώσετε την τεχνική έκθεση.